# Predicción de la alternativa elegida (A/B/C)

**Pregunta de la propuesta.** ¿Cuánto permite la exploración temprana del cursor anticipar la alternativa que la persona termina eligiendo?

**Unidad de análisis:** una decisión (persona × nodo). **Variable objetivo:** la alternativa elegida (A, B o C).

## Cómo se evita la trampa principal: la fuga de información
La última alternativa sobre la que pasa el cursor es **siempre** la elegida (se comprueba abajo, 100 %). Por eso cualquier variable que use el nodo completo (tiempo total, cuántas veces se miró cada opción, "elegió la más vista") contiene la respuesta. Para predecir de verdad hay que cortar el nodo en un instante `t`.

- **Ventana temporal absoluta:** se usan solo los eventos ocurridos en los primeros `t` segundos desde que apareció el nodo, con `t` = 2, 4, 6, 8 y 12 s. Esto responde la duda de la presentación sobre cómo definir la ventana.
- **Solo decisiones aún abiertas:** en el instante `t` se predice únicamente para quienes todavía no han decidido (duración del nodo > `t`). Es lo que se sabría en una aplicación real. Tanto el entrenamiento como la evaluación usan ese subconjunto, y las líneas de base se calculan sobre el mismo subconjunto.

## Modelos y evaluación (clases 1 a 3)
| Elemento | Detalle |
|---|---|
| Modelos de la propuesta | Regresión logística multinomial y árbol de clasificación (más un Random Forest como comparación) |
| Línea de base 1 | La alternativa más frecuente globalmente (A) |
| Línea de base 2 | La alternativa más frecuente **en ese nodo** (la que se conoce solo por saber el nodo) |
| Pregunta clave | ¿El proceso temprano aporta algo **más allá de conocer el nodo**? |
| Métricas | Exactitud, F1 macro, matriz de confusión y log-loss (mide la calidad de las probabilidades) |
| Validación | **Validación cruzada agrupada por persona** (5 grupos, repetida 3 veces): las 27 decisiones de una persona nunca quedan repartidas entre entrenamiento y prueba. Las líneas de base se calculan solo con los datos de entrenamiento de cada grupo. |
| Hiperparámetros | Se eligen con validación cruzada **interna** (también agrupada por persona), dentro de cada grupo de entrenamiento |
| Incertidumbre | Intervalos por bootstrap **sobre personas** (las decisiones de una misma persona no son independientes) y prueba de permutación |

**Tres conjuntos de variables:**
1. **Nodo:** cuál nodo es (línea de base).
2. **Proceso temprano:** hover acumulado por alternativa, veces que entró a cada una, cambios entre opciones, alternativas exploradas, primera alternativa mirada, alternativa bajo el cursor en `t` y alternativa con más hover hasta `t`.
3. **Historia previa de la persona:** qué eligió en los nodos anteriores (A/B/C) y qué tan lenta fue. Se construye solo con decisiones **anteriores** a la actual, así que es información disponible en el momento de predecir.

In [ ]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import f1_score, log_loss, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

SEED = 42
DATA = Path("../data") if Path("../data").exists() else Path("data")
OUT = Path("output")
OPC = ["A", "B", "C"]
IDX = {o: i for i, o in enumerate(OPC)}
VENTANAS = [2, 4, 6, 8, 12]          # segundos desde que aparece el nodo
R_CV, K_CV = 3, 5                    # repeticiones y grupos de la validación cruzada
B_BOOT = 2000

## 0. Carga y comprobaciones
`process_logs.csv` repite en cada fila los agregados y la lista completa de eventos del nodo, así que se deja una fila por persona y nodo. Los tiempos de los eventos son absolutos y se convierten a milisegundos desde el inicio del nodo.

In [ ]:
logs = pd.read_csv(DATA / "process_logs.csv",
                   usecols=["session_id", "user_id", "node_id", "start_time", "total_duration", "final_choice", "events"])
nodo = logs.drop_duplicates(["session_id", "node_id"]).reset_index(drop=True)
dec = pd.read_csv(DATA / "explicit_decisions.csv", usecols=["user_id", "node_id", "decision_order", "option_id"])
nodo = nodo.merge(dec, on=["user_id", "node_id"], how="left", validate="one_to_one")
assert len(nodo) == 1636 and nodo["decision_order"].notna().all()
assert (nodo["final_choice"] == nodo["option_id"]).all()

def parsear(js, t0):
    return [(e["timestamp"] - t0, e["type"], e["metadata"]["option_id"]) for e in json.loads(js)]
nodo["ev"] = [parsear(j, s) for j, s in zip(nodo["events"], nodo["start_time"])]
nodo["duracion_ms"] = nodo["total_duration"]
nodo["y"] = nodo["final_choice"].map(IDX)

# Comprobación de la fuga: la última opción tocada por el cursor es la elegida
ultima = np.array([ev[-1][2] for ev in nodo["ev"]])
print(f"La última alternativa tocada por el cursor coincide con la elegida en el {np.mean(ultima == nodo['final_choice']):.1%} de las decisiones")
print(f"{nodo['user_id'].nunique()} personas | {nodo['node_id'].nunique()} nodos | {len(nodo)} decisiones")
print("Distribución de la alternativa elegida:", nodo["final_choice"].value_counts(normalize=True).round(3).to_dict())
print("Eventos fuera del orden temporal:", sum(np.any(np.diff([t for t, _, _ in ev]) < 0) for ev in nodo["ev"]))
print("Fracción de decisiones aún abiertas en el instante t:",
      {t: round(float((nodo["duracion_ms"] > t * 1000).mean()), 3) for t in VENTANAS})

## 1. Variables

### 1A. Proceso temprano (hasta el instante `t`)
Se recorren los eventos en orden y se cortan en `t`. Si el cursor sigue sobre una alternativa en `t`, ese tiempo se cuenta hasta `t` (no hasta el final del nodo).

In [ ]:
def features_ventana(ev, t_ms):
    hover = dict.fromkeys(OPC, 0.0)
    entra = dict.fromkeys(OPC, 0)
    abierto, primera, ultima_ent, cambios = None, None, None, 0
    for t, tipo, o in ev:
        if t > t_ms:
            break
        if tipo == "hover_enter":
            if abierto is not None:
                hover[abierto[0]] += t - abierto[1]
            entra[o] += 1
            if ultima_ent is not None and o != ultima_ent:
                cambios += 1
            if primera is None:
                primera = o
            abierto, ultima_ent = (o, t), o
        elif abierto is not None and abierto[0] == o:
            hover[o] += t - abierto[1]
            abierto = None
    cursor = None
    if abierto is not None:
        hover[abierto[0]] += t_ms - abierto[1]
        cursor = abierto[0]
    lider = max(OPC, key=hover.get) if sum(hover.values()) > 0 else None
    fila = {}
    for o in OPC:
        fila[f"hover_{o}_s"] = hover[o] / 1000
        fila[f"entra_{o}"] = entra[o]
    fila["n_cambios"] = cambios
    fila["n_exploradas"] = sum(h > 0 for h in hover.values())
    for o in OPC:
        fila[f"primera_{o}"] = float(primera == o)
        fila[f"cursor_{o}"] = float(cursor == o)
        fila[f"lider_{o}"] = float(lider == o)
    fila["_cursor"], fila["_lider"] = cursor, lider
    return fila

PROC = [f"hover_{o}_s" for o in OPC] + [f"entra_{o}" for o in OPC] + ["n_cambios", "n_exploradas"] \
     + [f"{p}_{o}" for p in ["primera", "cursor", "lider"] for o in OPC]

### 1B. Historia previa de la persona (causal)
Para cada decisión se usan solo las decisiones **anteriores** de la misma persona (ordenadas por `decision_order`): proporción de A, B y C elegidas (suavizada con un +1 para evitar ceros), número de decisiones previas y el ritmo promedio en esos nodos. El ritmo se mide con el tiempo de cada nodo estandarizado entre las 60 personas (usa solo tiempos, nunca la alternativa elegida).

In [ ]:
nodo = nodo.sort_values(["user_id", "decision_order"]).reset_index(drop=True)
nodo["log_dur"] = np.log(nodo["duracion_ms"] / 1000)
nodo["z_dur"] = nodo.groupby("node_id")["log_dur"].transform(lambda s: (s - s.mean()) / (s.std(ddof=0) or 1))
g = nodo.groupby("user_id")
nodo["n_prev"] = g.cumcount()
for o in OPC:
    c = (nodo["final_choice"] == o).astype(int)
    nodo[f"prev_{o}"] = (c.groupby(nodo["user_id"]).cumsum() - c + 1) / (nodo["n_prev"] + 3)
nodo["prev_z_dur"] = ((nodo.groupby("user_id")["z_dur"].cumsum() - nodo["z_dur"]) / nodo["n_prev"].replace(0, np.nan)).fillna(0)
HIST = ["n_prev", "prev_A", "prev_B", "prev_C", "prev_z_dur"]
print("Historia previa (primeras filas de una persona):")
nodo.loc[nodo["user_id"] == nodo["user_id"].iloc[0], ["decision_order", "final_choice"] + HIST].head(5)

In [ ]:
def tabla_ventana(t):
    """Una fila por decisión aún abierta en el instante t, con las variables del proceso hasta t."""
    abiertas = nodo["duracion_ms"] > t * 1000
    f = pd.DataFrame([features_ventana(ev, t * 1000) for ev in nodo.loc[abiertas, "ev"]], index=nodo.index[abiertas])
    return nodo.loc[abiertas, ["user_id", "node_id", "y", "final_choice"] + HIST].join(f)

TABLAS = {t: tabla_ventana(t) for t in VENTANAS}
print({t: len(df) for t, df in TABLAS.items()})
TABLAS[VENTANAS[2]][["node_id", "final_choice"] + PROC[:8]].head(4)

### 1C. Reglas simples sin entrenamiento
Antes de entrenar nada: ¿qué tan bien acierta la regla "elige la alternativa donde está el cursor en `t`" o "elige la que más ha mirado hasta `t`"? Si no hay ninguna alternativa (el cursor está entre opciones), se predice A, la más frecuente.

In [ ]:
filas = []
for t, df in TABLAS.items():
    cur = df["_cursor"].fillna("A"); lid = df["_lider"].fillna("A")
    filas.append({"t_s": t, "n": len(df),
                  "acc_regla_cursor": (cur == df["final_choice"]).mean(),
                  "acc_regla_mas_mirada": (lid == df["final_choice"]).mean(),
                  "acc_siempre_A": (df["final_choice"] == "A").mean(),
                  "sin_hover_en_t": (df["_cursor"].isna()).mean()})
reglas = pd.DataFrame(filas).set_index("t_s")
reglas

## 2. Modelos y validación cruzada agrupada por persona

- Cada **repetición** reparte las 60 personas al azar en 5 grupos. Las mismas particiones se usan en todas las ventanas y modelos, para que sean comparables.
- **Regresión logística multinomial:** el parámetro de regularización `C` se elige por validación cruzada interna agrupada por persona, con pérdida logarítmica. Las variables numéricas se estandarizan dentro del entrenamiento.
- **Árbol:** profundidad máxima y tamaño mínimo de hoja, elegidos igual que `C`.
- **Random Forest:** 200 árboles, hojas de al menos 5 decisiones, sin ajuste de hiperparámetros.
- **Moda por nodo:** la línea de base 2. Predice, en cada nodo, la alternativa más frecuente en el entrenamiento (con suavizado de Laplace para obtener probabilidades).

In [ ]:
def folds_por_repeticion(usuarios, r, K=K_CV):
    rng = np.random.default_rng(1000 + r)
    perm = rng.permutation(len(usuarios))
    fold = np.empty(len(usuarios), int)
    fold[perm] = np.arange(len(usuarios)) % K
    return dict(zip(usuarios, fold))

USUARIOS = sorted(nodo["user_id"].unique())
FOLDS = [folds_por_repeticion(USUARIOS, r) for r in range(max(R_CV, 1))]

class ModaGlobal:
    def fit(self, X, y):
        self.p = (np.bincount(y, minlength=3) + 1) / (len(y) + 3); return self
    def predict_proba(self, X):
        return np.tile(self.p, (len(X), 1))

class ModaPorNodo:
    def fit(self, X, y):
        self.tab = (pd.crosstab(X["node_id"].values, y).reindex(columns=[0, 1, 2], fill_value=0) + 1).astype(float)
        self.prior = np.bincount(y, minlength=3) + 1.0
        return self
    def predict_proba(self, X):
        p = self.tab.reindex(X["node_id"].values).values
        p = np.where(np.isnan(p), self.prior, p)
        return p / p.sum(1, keepdims=True)

def preproceso(cols):
    pasos = [("nodo", OneHotEncoder(handle_unknown="ignore"), ["node_id"])]
    if cols:
        pasos.append(("num", StandardScaler(), cols))
    return ColumnTransformer(pasos)

GRID_LR = {"clf__C": np.logspace(-2, 1, 5)}
GRID_ARBOL = {"clf__max_depth": [2, 3, 4, 6], "clf__min_samples_leaf": [10, 25, 50]}
MODELOS = {
    "Global (moda)": dict(tipo="global"),
    "Nodo (moda por nodo)": dict(tipo="nodo"),
    "LR nodo+historia": dict(tipo="sk", cols=HIST, grid=GRID_LR,
                             clf=lambda: LogisticRegression(max_iter=3000)),
    "LR nodo+proceso": dict(tipo="sk", cols=PROC, grid=GRID_LR,
                            clf=lambda: LogisticRegression(max_iter=3000)),
    "Árbol nodo+proceso": dict(tipo="sk", cols=PROC, grid=GRID_ARBOL,
                               clf=lambda: DecisionTreeClassifier(random_state=SEED)),
    "RF nodo+proceso": dict(tipo="sk", cols=PROC, grid=None,
                            clf=lambda: RandomForestClassifier(200, min_samples_leaf=5, random_state=SEED, n_jobs=1)),
    "LR nodo+proceso+historia": dict(tipo="sk", cols=PROC + HIST, grid=GRID_LR,
                                     clf=lambda: LogisticRegression(max_iter=3000)),
}

def ajustar(spec, Xtr, ytr, gtr):
    if spec["tipo"] == "global":
        return ModaGlobal().fit(Xtr, ytr)
    if spec["tipo"] == "nodo":
        return ModaPorNodo().fit(Xtr, ytr)
    pipe = Pipeline([("pre", preproceso(spec["cols"])), ("clf", spec["clf"]())])
    if spec["grid"] is None:
        return pipe.fit(Xtr, ytr)
    cv = list(GroupKFold(3).split(Xtr, ytr, gtr))
    return GridSearchCV(pipe, spec["grid"], cv=cv, scoring="neg_log_loss").fit(Xtr, ytr)

def cv_agrupada(df, y, nombres, reps):
    """Devuelve, para cada modelo, las probabilidades fuera de muestra (repeticiones × decisiones × 3)."""
    n = len(df)
    P = {m: np.zeros((len(reps), n, 3)) for m in nombres}
    usr = df["user_id"].values
    for i, r in enumerate(reps):
        fold = np.array([FOLDS[r][u] for u in usr])
        for k in range(K_CV):
            tr, te = fold != k, fold == k
            for m in nombres:
                est = ajustar(MODELOS[m], df[tr], y[tr], usr[tr])
                P[m][i, te] = est.predict_proba(df[te])
    return P

t0 = time.time()
RESULT = {}
for t in VENTANAS:
    df = TABLAS[t]
    RESULT[t] = cv_agrupada(df, df["y"].values, list(MODELOS), list(range(R_CV)))
    print(f"ventana {t:>2} s: {len(df)} decisiones | {time.time() - t0:.0f} s transcurridos")

## 3. Resultados

- **Exactitud y F1 macro:** se calculan por repetición y se promedian.
- **Log-loss:** más bajo es mejor. Mide si las *probabilidades* son buenas, no solo la alternativa más probable, y es más sensible que la exactitud.
- **Intervalos del 95 %** por bootstrap sobre personas: se remuestrean las 60 personas con reemplazo y se recalcula la diferencia con la moda por nodo. Si el intervalo de la diferencia incluye 0, no hay evidencia de mejora.

In [ ]:
def metricas(P, y):
    pred = P.argmax(2)
    acc = np.mean([(p == y).mean() for p in pred])
    f1 = np.mean([f1_score(y, p, average="macro", labels=[0, 1, 2], zero_division=0) for p in pred])
    ll = np.mean([log_loss(y, np.clip(p, 1e-9, 1), labels=[0, 1, 2]) for p in P])
    return acc, f1, ll

def por_decision(P, y):
    """Acierto y pérdida logarítmica de cada decisión, promediados sobre las repeticiones."""
    ok = (P.argmax(2) == y).mean(0)
    ll = -np.log(np.clip(P[:, np.arange(len(y)), y], 1e-9, 1)).mean(0)
    return ok, ll

def bootstrap_personas(d, usuarios, B=B_BOOT, seed=SEED):
    """IC95 % de la media de d (por decisión) remuestreando personas."""
    rng = np.random.default_rng(seed)
    cod = pd.factorize(usuarios)[0]
    sumas, cuentas = np.bincount(cod, weights=d), np.bincount(cod)
    sorteo = rng.integers(0, len(sumas), (B, len(sumas)))
    est = sumas[sorteo].sum(1) / cuentas[sorteo].sum(1)
    return np.percentile(est, [2.5, 97.5])

filas = []
for t in VENTANAS:
    df = TABLAS[t]; y = df["y"].values; usr = df["user_id"].values
    base_ok, base_ll = por_decision(RESULT[t]["Nodo (moda por nodo)"], y)
    for m, P in RESULT[t].items():
        acc, f1, ll = metricas(P, y)
        ok, lld = por_decision(P, y)
        d_acc, d_ll = ok - base_ok, base_ll - lld            # positivo = mejor que la moda por nodo
        ic_acc, ic_ll = bootstrap_personas(d_acc, usr), bootstrap_personas(d_ll, usr)
        filas.append({"t_s": t, "n": len(df), "modelo": m, "acc": acc, "f1_macro": f1, "logloss": ll,
                      "dif_acc": d_acc.mean(), "ic_acc_inf": ic_acc[0], "ic_acc_sup": ic_acc[1],
                      "dif_ll": d_ll.mean(), "ic_ll_inf": ic_ll[0], "ic_ll_sup": ic_ll[1]})
res = pd.DataFrame(filas)
print("Exactitud, F1 macro y log-loss por ventana y modelo (promedio de las 3 repeticiones):")
res.pivot(index="modelo", columns="t_s", values="acc").loc[list(MODELOS)]

In [ ]:
print("F1 macro:")
display(res.pivot(index="modelo", columns="t_s", values="f1_macro").loc[list(MODELOS)])
print("Log-loss (más bajo es mejor):")
res.pivot(index="modelo", columns="t_s", values="logloss").loc[list(MODELOS)]

In [ ]:
# Mejora respecto de la moda por nodo, con intervalo del 95 % (bootstrap sobre personas)
comp = [m for m in MODELOS if not m.startswith(("Global", "Nodo"))]
colores = dict(zip(comp, sns.color_palette("tab10", len(comp))))
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for m in comp:
    s = res[res.modelo == m].set_index("t_s")
    off = (comp.index(m) - len(comp) / 2) * 0.12
    axes[0].errorbar(np.arange(len(VENTANAS)) + off, s["dif_acc"] * 100, yerr=[(s["dif_acc"] - s["ic_acc_inf"]) * 100, (s["ic_acc_sup"] - s["dif_acc"]) * 100],
                     fmt="o-", color=colores[m], label=m, capsize=3, ms=5)
    axes[1].errorbar(np.arange(len(VENTANAS)) + off, s["dif_ll"], yerr=[s["dif_ll"] - s["ic_ll_inf"], s["ic_ll_sup"] - s["dif_ll"]],
                     fmt="o-", color=colores[m], capsize=3, ms=5)
for ax, tit, yl in [(axes[0], "Exactitud", "puntos porcentuales"), (axes[1], "Log-loss", "reducción (positivo = mejor)")]:
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(range(len(VENTANAS))); ax.set_xticklabels([f"{t} s" for t in VENTANAS])
    ax.set_title(f"{tit}: mejora sobre la moda por nodo"); ax.set_ylabel(yl); ax.set_xlabel("ventana de observación")
axes[0].legend(fontsize=7)
axes[2].plot(range(len(VENTANAS)), [(nodo["duracion_ms"] > t * 1000).mean() * 100 for t in VENTANAS], "o-", color="#555555")
axes[2].set_xticks(range(len(VENTANAS))); axes[2].set_xticklabels([f"{t} s" for t in VENTANAS])
axes[2].set_ylabel("% de decisiones"); axes[2].set_title("Decisiones aún abiertas en el instante t"); axes[2].set_xlabel("ventana de observación")
plt.tight_layout(); plt.show()

res[res.modelo.isin(comp)].pivot(index="modelo", columns="t_s", values="dif_acc").loc[comp].mul(100).round(2)

## 4. Prueba de permutación: ¿el proceso temprano aporta algo?

Los intervalos anteriores dependen de la elección del modelo y de la ventana. La prueba de permutación responde directamente: **si no hubiera relación entre el proceso y la alternativa elegida, ¿qué mejora sobre la moda por nodo se vería solo por azar?**

Se mezclan las alternativas elegidas **dentro de cada nodo** (esto conserva la frecuencia de cada alternativa en cada nodo, así que la línea de base no cambia) y se repite todo el procedimiento con una repetición de validación cruzada. Se compara la regresión logística con proceso temprano contra la moda por nodo, midiendo la reducción del log-loss y la mejora de la exactitud.

**Cómo leer la tabla.**
- Con solo 40 permutaciones, el menor p-valor posible es 1/41 = 0,024.
- Con las alternativas mezcladas, el modelo con muchas variables **empeora** el log-loss respecto de la moda por nodo porque ajusta ruido, así que el nulo queda por debajo de 0 (`nulo_p95` negativo). Un p bajo en log-loss con diferencia observada negativa significa "menos malo que el ruido", no "mejor que la moda por nodo". La mejora real sobre la línea de base se lee en la diferencia observada.

In [ ]:
N_PERM = 40
def delta_ll(df, y):
    P = cv_agrupada(df, y, ["Nodo (moda por nodo)", "LR nodo+proceso"], [0])
    base, mod = metricas(P["Nodo (moda por nodo)"], y), metricas(P["LR nodo+proceso"], y)
    return base[2] - mod[2], mod[0] - base[0]

rng = np.random.default_rng(SEED)
filas, nulos = [], {}
t0 = time.time()
for t in VENTANAS:
    df = TABLAS[t]; y = df["y"].values
    obs_ll, obs_acc = delta_ll(df, y)
    nulo = []
    idx_nodo = [np.where(df["node_id"].values == n)[0] for n in df["node_id"].unique()]
    for _ in range(N_PERM):
        yp = y.copy()
        for ix in idx_nodo:
            yp[ix] = rng.permutation(y[ix])
        nulo.append(delta_ll(df, yp))
    nulo = np.array(nulo); nulos[t] = (obs_ll, nulo[:, 0])
    filas.append({"t_s": t, "dif_logloss_obs": obs_ll, "nulo_p95": np.percentile(nulo[:, 0], 95),
                  "p_logloss": (np.sum(nulo[:, 0] >= obs_ll) + 1) / (N_PERM + 1),
                  "dif_acc_obs": obs_acc, "p_acc": (np.sum(nulo[:, 1] >= obs_acc) + 1) / (N_PERM + 1)})
    print(f"ventana {t:>2} s listo | {time.time() - t0:.0f} s")
perm = pd.DataFrame(filas).set_index("t_s")
perm

## 5. Detalle del mejor caso

Se elige la ventana en que la regresión logística con proceso temprano reduce más el log-loss respecto de la moda por nodo. Es una elección posterior a ver los resultados, así que las cifras de esta sección son descriptivas.

In [ ]:
sub = res[res.modelo == "LR nodo+proceso"].set_index("t_s")
T_MEJOR = int(sub["dif_ll"].idxmax())
print(f"Ventana elegida: {T_MEJOR} s (reducción de log-loss = {sub.loc[T_MEJOR, 'dif_ll']:.4f}, IC95 % [{sub.loc[T_MEJOR, 'ic_ll_inf']:.4f}, {sub.loc[T_MEJOR, 'ic_ll_sup']:.4f}])")
dfm = TABLAS[T_MEJOR]; ym = dfm["y"].values

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, m in zip(axes, ["Nodo (moda por nodo)", "LR nodo+proceso"]):
    cm = confusion_matrix(ym, RESULT[T_MEJOR][m][0].argmax(1), labels=[0, 1, 2])
    sns.heatmap(cm / cm.sum(1, keepdims=True), annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=1, ax=ax, cbar=False,
                xticklabels=OPC, yticklabels=OPC)
    ax.set_xlabel("predicha"); ax.set_ylabel("elegida"); ax.set_title(f"{m} ({T_MEJOR} s)\n(color = % de cada fila, número = decisiones)")
plt.tight_layout(); plt.show()

In [ ]:
# ¿En qué nodos ayuda el proceso temprano? (con ~60 decisiones por nodo, cada barra es muy ruidosa)
ok_base, _ = por_decision(RESULT[T_MEJOR]["Nodo (moda por nodo)"], ym)
ok_mod, _ = por_decision(RESULT[T_MEJOR]["LR nodo+proceso"], ym)
por_nodo = pd.DataFrame({"node_id": dfm["node_id"].values, "base": ok_base, "modelo": ok_mod}).groupby("node_id").agg(
    n=("base", "size"), acc_nodo=("base", "mean"), acc_modelo=("modelo", "mean"))
por_nodo["dif"] = por_nodo["acc_modelo"] - por_nodo["acc_nodo"]
por_nodo = por_nodo.sort_values("dif")
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh([n.replace("MLQ5X_", "") for n in por_nodo.index], por_nodo["dif"] * 100,
        color=["#C44E52" if d < 0 else "#4C72B0" for d in por_nodo["dif"]])
ax.axvline(0, color="black", lw=0.8); ax.set_xlabel("puntos porcentuales de exactitud (modelo − moda por nodo)")
ax.set_title(f"Diferencia por nodo, ventana de {T_MEJOR} s (≈ 60 decisiones por nodo)")
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout(); plt.show()
print(f"Nodos donde el modelo supera a la moda: {(por_nodo['dif'] > 0).sum()} | empata: {(por_nodo['dif'] == 0).sum()} | pierde: {(por_nodo['dif'] < 0).sum()}")

### Qué variables usa el modelo
Importancia por permutación **en los datos de prueba** de cada grupo de la validación cruzada: se mezclan las filas de un bloque de variables y se mide cuánto empeora el log-loss. Los bloques se mezclan completos (por ejemplo, las tres columnas de hover por alternativa juntas), porque varias variables están muy relacionadas entre sí y mezclarlas por separado subestimaría su importancia.

Si el modelo no mejora a la moda por nodo, estas importancias describen un modelo que no predice mejor y no deben interpretarse.

In [ ]:
BLOQUES = {
    "hover acumulado por alternativa": [f"hover_{o}_s" for o in OPC] + [f"entra_{o}" for o in OPC],
    "cambios y exploración": ["n_cambios", "n_exploradas"],
    "primera alternativa mirada": [f"primera_{o}" for o in OPC],
    "alternativa bajo el cursor en t": [f"cursor_{o}" for o in OPC],
    "alternativa más mirada hasta t": [f"lider_{o}" for o in OPC],
}
rng = np.random.default_rng(SEED)
fold = np.array([FOLDS[0][u] for u in dfm["user_id"].values])
imp = {b: [] for b in BLOQUES}
for k in range(K_CV):
    tr, te = fold != k, fold == k
    est = ajustar(MODELOS["LR nodo+proceso"], dfm[tr], ym[tr], dfm.loc[tr, "user_id"].values)
    Xte = dfm[te].copy()
    base_ll = log_loss(ym[te], est.predict_proba(Xte), labels=[0, 1, 2])
    for b, cols in BLOQUES.items():
        for _ in range(20):
            Xp = Xte.copy()
            Xp[cols] = Xte[cols].values[rng.permutation(len(Xte))]
            imp[b].append(log_loss(ym[te], est.predict_proba(Xp), labels=[0, 1, 2]) - base_ll)
importancia = pd.DataFrame({"aumento_logloss": {b: np.mean(v) for b, v in imp.items()},
                            "sd": {b: np.std(v) for b, v in imp.items()}}).sort_values("aumento_logloss", ascending=False)
importancia

## 6. Guardar resultados

In [ ]:
OUT.mkdir(exist_ok=True)
res.to_csv(OUT / "resultados_prediccion_alternativas.csv", index=False)
print("Guardado en", (OUT / "resultados_prediccion_alternativas.csv").resolve())

## 7. Conclusiones (ejecución del 26-11-2026, semilla 42)

**1. Conocer el nodo ya explica gran parte de la elección.** La alternativa más frecuente en cada nodo acierta entre 60 % y 62 % (F1 macro ≈ 0,56), frente a 48 % a 50 % de la más frecuente global (A). Esa es la línea de base real contra la que hay que medir el proceso del cursor.

**2. El proceso de los primeros segundos no anticipa la elección más allá del nodo.**
- Con ventanas de 2, 4, 6 y 8 s, la regresión logística con proceso temprano no mejora la exactitud de forma consistente: la diferencia va de −0,2 a +1,2 puntos, y solo en 6 s el intervalo de confianza excluye el 0 ([0,1; 2,4]).
- En 2 y 4 s las probabilidades son **peores** que las de la moda por nodo (log-loss, intervalos que excluyen el 0).
- Una razón es que casi no hay información: a los 2 s el 88 % de las personas aún no ha puesto el cursor sobre ninguna alternativa, y a los 4 s el 80 %.
- La regla simple "elige donde está el cursor" acierta 48 % a 49 %, igual que elegir siempre A.

**3. Con 12 s aparece una mejora modesta.**
- La regresión logística con proceso llega a 63,7 % de exactitud frente a 60,2 % de la moda por nodo (+3,5 puntos; IC 95 % [1,9; 5,1]) y el F1 macro sube de 0,56 a 0,61.
- El log-loss casi no cambia (0,795 frente a 0,803; IC [−0,009; 0,023]): las probabilidades no mejoran de forma clara.
- Los 12 s no son "temprano": la mediana de un nodo es de 18 s y esta ventana ya deja fuera al 22 % de las decisiones, que se toman antes.

**4. La prueba de permutación va en la misma dirección.** Para la exactitud, p = 0,024 a 6 s y 12 s (el mínimo posible con 40 permutaciones) y 0,049 a 8 s; no hay señal a 2 y 4 s. Con una corrección de Bonferroni por las 5 ventanas, ninguna llega a 0,05. El p de log-loss no se lee como mejora (ver la nota de la sección 4).

**5. La elección del modelo importa.**
- La regresión logística supera al Random Forest y este al árbol de clasificación.
- El árbol es el peor en todas las ventanas (−0,7 a −5,4 puntos de exactitud y un log-loss mucho peor): con ≈ 1.300 a 1.600 decisiones de 60 personas sobreajusta.
- El Random Forest no mejora a la logística y empeora el log-loss.
- La historia previa de la persona (qué eligió antes y qué tan lenta fue) no aporta: como máximo +0,6 puntos, con el intervalo incluyendo el 0.

**6. Qué usa el modelo (descriptivo, ventana de 12 s).** Al mezclar cada bloque de variables, el log-loss aumenta más con el hover acumulado por alternativa (0,050) y con la alternativa bajo el cursor en `t` (0,039), después con los cambios y la exploración (0,021). La primera alternativa mirada y la más mirada hasta `t` casi no importan (≈ 0,006). Por nodo, el modelo supera a la moda en 13 de 28 nodos, empata en 6 y pierde en 9; con ≈ 60 decisiones por nodo cada barra es muy ruidosa.

**7. Cautelas.**
- **Ventana elegida después de ver los resultados.** Los 12 s se escogen entre 5 ventanas por ser la de mayor mejora; los intervalos de confianza no corrigen esa elección.
- **Cada ventana es otra población.** En 12 s solo quedan las decisiones aún abiertas, que son más lentas, y la línea de base cambia ligeramente de una ventana a otra.
- **Pocas personas.** Son 60, y las decisiones de una misma persona no son independientes. Por eso la validación y los intervalos se hacen por persona.
- **Las alternativas A/B/C no significan lo mismo en cada nodo.** La regresión multinomial con variables compartidas entre nodos supone que, por ejemplo, el hover sobre "A" tiene un efecto parecido en todos. Un modelo de elección discreta (con una variable por alternativa y un solo coeficiente) sería más natural y queda como trabajo futuro.
- **Capacidad predictiva, no causalidad.** Que el cursor se detenga sobre una alternativa no la causa.

**8. Respuesta a la pregunta de la propuesta.** La exploración **temprana** del cursor (hasta 8 s) no permite anticipar la alternativa elegida más allá de conocer el nodo. Observando hasta 12 s, la mejora existe pero es pequeña (≈ 3,5 puntos de exactitud) y no resiste del todo una corrección por la elección de la ventana. Esto es coherente con el notebook 02, donde la conducta del cursor aparece como un continuo y no como tipos claros, y con el 01, donde la telemetría solo se asocia débilmente con el MLQ externo.